In [0]:
df = spark.read.csv("/Volumes/forecasting/default/temperature/IOT-temp.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("/Volumes/forecasting/default/temperature/IOT-temp.parquet")

In [0]:
df2 = spark.read.table("forecasting.default.iot_temp_2")
display(df2)

In [0]:
from pyspark.sql.functions import to_timestamp

df = df2.withColumn(
    "noted_date",
    to_timestamp("noted_date", "dd-MM-yyyy HH:mm")
)

In [0]:
df = df.filter(df["out/in"] == "In")

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# Ordenar por fecha
df = df.orderBy("noted_date")

# Convertir fecha a timestamp numérico para usar como feature
df = df.withColumn("noted_date_ts", df["noted_date"].cast("long"))

# Preparar features
assembler = VectorAssembler(inputCols=["noted_date_ts"], outputCol="features")
df_features = assembler.transform(df).select("features", "temp")

# Entrenar modelo de regresión lineal
lr = LinearRegression(featuresCol="features", labelCol="temp")
model = lr.fit(df_features)

# Predecir valores futuros (ejemplo: próximos 5 minutos)
from pyspark.sql.functions import lit
import datetime

last_date = df.agg({"noted_date": "max"}).collect()[0][0]
future_dates = [last_date + datetime.timedelta(minutes=i) for i in range(1, 6)]
future_df = spark.createDataFrame([(d,) for d in future_dates], ["noted_date"])
future_df = future_df.withColumn("noted_date_ts", future_df["noted_date"].cast("long"))
future_features = assembler.transform(future_df).select("features", "noted_date")

predictions = model.transform(future_features)
display(predictions.select("noted_date", "prediction"))